# App1 ReAct Agent：让 LLM 使用工具 教案

**课程名称：** ReAct Agent：让 LLM 使用工具

**预计总时长：** 75-85 分钟

**源文件：** `Applications/App1_ReAct_Agent.ipynb`（共 18 个 Cell，含 Markdown + Code）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 环境配置 + ReAct 概念导入 | Cell 0-3（intro-cell ~ setup-llm） | 8 min |
| 8-25 min | 工具系统：Tool 类 + Function Calling 格式 + JSON Schema | Cell 4-8（tools-header ~ 8igrjqox0qg） | 17 min |
| 25-40 min | 四个工具实现 + 测试验证 | Cell 9-14（tool-implementations ~ 8igrjqox0qg） | 15 min |
| 40-45 min | **休息 + 回顾** | -- | 5 min |
| 45-60 min | ReAct Agent 核心实现 + 解析逻辑 | Cell 15-16（agent-header ~ agent-class） | 15 min |
| 60-75 min | Agent 测试：数学 / 时间 / 天气 / 搜索 / 复合问题 | Cell 17-22（test-header ~ test-complex） | 15 min |
| 75-80 min | 可视化 + 总结 + 练习 | Cell 23-25（viz-header ~ exercise） | 5 min |
| 80-85 min | **休息 + 总结回顾** | -- | 5 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 已安装
- [ ] 确认依赖已安装：`requests`, `matplotlib`, `json`, `re`, `dataclasses`
- [ ] 确认 LLM 后端至少一种可用（Ollama 本地 / DashScope API / OpenAI API）
- [ ] 如用 Ollama：确认已安装并运行 `ollama pull qwen3:4b`，参考 `PREPARE_OLLAMA.ipynb`
- [ ] 如用 DashScope：确认 `DASHSCOPE_API_KEY` 已设置
- [ ] 确认网络畅通（天气工具调用 Open-Meteo API，搜索工具调用 DuckDuckGo API）
- [ ] 预跑一遍全部 Cell，确认所有输出正常
- [ ] 确认 `utils/llm_backend.py` 模块存在（统一 LLM 后端接口）
- [ ] 准备白板或画板，用于手绘 ReAct 循环图

---

## 第一段：开场 + 环境配置 + ReAct 概念导入（Cell intro-cell ~ setup-llm）

📍 运行 Cell intro-cell、875ivuk0y2u（Markdown 导读），Cell 6ed0d721（Ollama 提醒），Cell setup-imports（环境准备代码），Cell setup-llm（LLM 后端配置）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：从 Ch12 的理论模拟到真实 LLM Agent 的飞跃
- 理解 ReAct = Reasoning + Acting 的核心思想
- 确认环境就绪，LLM 后端连通

🗣 讲课话术

> 大家好！上节课 Ch12 我们用 if-else 规则模拟了 Agent 的 ReAct 循环，大家可能会觉得「这也不是真正的 AI 在思考吧？」今天我们要做一件激动人心的事——**用真实的 LLM 来驱动 Agent**。不再是规则模拟，而是模型真正在「想」该调什么工具。
>
> 先看 Cell intro-cell 的目标。三件事：(1) 用真实 LLM（Ollama / DashScope / OpenAI 都行）；(2) 实现完整的 ReAct 循环；(3) 做一个可扩展的工具系统。
>
> 什么是 ReAct？看 Cell intro-cell 的流程图——`问题 → 思考(Thought) → 行动(Action) → 观察(Observation) → 思考 → ... → 答案`。跟上节课一样的循环，但这次是真正的 LLM 在每一步做决策。
>
> 我们有四个真实工具——计算器用安全的 Python eval，搜索用 DuckDuckGo API，日期时间用系统时间，天气用 Open-Meteo API。全是真实的，不是模拟！
>
> Cell 875ivuk0y2u 告诉我们学习目标和前置知识——需要 Ch12 的 Agent & RAG 原理，预计 40 分钟跑完代码，但我们加上讲解大约 80 分钟。
>
> 好，运行 Cell setup-imports。（运行）看到 `[OK] 环境准备完成!` 就好。注意这里导入了 `utils.llm_backend` 模块——这是一个统一接口，不管你用 Ollama、DashScope 还是 OpenAI，调用方式都一样。
>
> 运行 Cell setup-llm。（运行）这段代码会按优先级自动检测后端——先试 Ollama，再试 DashScope，最后试 OpenAI。看输出，我们用的是哪个后端？（指向输出）好的，连接成功！如果大家在自己电脑上跑，按注释里的三种方式任选一种配置即可。

👀 输出要点
- Cell setup-imports：`[OK] 环境准备完成!`
- Cell setup-llm：`[OK] 使用 XXX 后端: OK...`（XXX 为 Ollama / 通义千问 / OpenAI）
- 如果全部失败：会显示三种配置方式的指引

❓ 预判问题
- **Q：Ollama 和 DashScope 有什么区别？**
  A：Ollama 是本地推理，模型跑在你自己电脑上，免费但需要 GPU 或较强 CPU。DashScope 是阿里云的 API 服务，模型在云端，需要 API Key 但不吃本地资源。教学推荐 DashScope（稳定、快），个人练习推荐 Ollama（免费、隐私）。
- **Q：qwen3:4b 是什么模型？**
  A：通义千问 3 代的 4B 参数版本，适合在消费级 GPU 或 CPU 上运行。参数量小但对工具调用场景够用了。
- **Q：环境报错 ModuleNotFoundError 怎么办？**
  A：`pip install requests matplotlib`，然后重启 Kernel。

➡️ 转场

> 环境好了，LLM 连通了。下面进入核心内容——怎么定义工具，让 LLM 知道它有哪些能力可以用？

---

## 第二段：工具系统——Tool 类 + Function Calling 格式（Cell tools-header ~ 9gpup7a0edc）

📍 浏览 Cell tools-header（Markdown），运行 Cell tool-class（Tool 数据类），浏览 Cell z1nksdmyc8（Function Calling 说明），运行 Cell 9gpup7a0edc（Function Calling JSON 格式演示）

⏱ 时间分配：17 分钟（Tool 类 5 分钟 + Function Calling 12 分钟）

🎯 本段目标
- 理解工具的四要素：名称、描述、参数（JSON Schema）、执行函数
- 掌握 Function Calling 的三步流程和 JSON 格式
- 理解 Function Calling vs ReAct 文本解析的区别

🗣 讲课话术

> 先看 Cell tools-header 的说明。每个工具需要四样东西——名称（唯一标识符）、描述（LLM 用来决定何时使用）、参数（JSON Schema 格式）、执行函数（实际的功能实现）。
>
> 运行 Cell tool-class。（运行）这是一个 `@dataclass`，很简洁。重点看两个方法：
> - `run(**kwargs)`：执行工具，内部调用 `self.func`，有 try-except 兜底
> - `to_openai_format()`：把我们的工具定义转换成 OpenAI API 期望的 JSON 格式
>
> 打个比方：`Tool` 类就像一个「工具说明书」，上面写了工具叫什么名字、能干什么、需要什么输入参数。LLM 看了说明书才知道该怎么用这个工具。
>
> 现在看 Cell z1nksdmyc8 的重要概念——**Function Calling**。这是现代 LLM 调用工具的标准方式。核心区别是什么？（等 2 秒）
>
> 看这个对比表：
> - **文本解析（ReAct 方式）**：LLM 输出 `Action: calculator\nAction Input: {"expression": "2+2"}`，我们用正则表达式解析。优点是通用，缺点是 LLM 可能格式写错。
> - **Function Calling**：LLM 直接输出合法 JSON `{"name": "calculator", "arguments": {...}}`。优点是可靠，缺点是只有部分模型支持。
>
> 我们这个 notebook 用的是文本解析方式——更通用，兼容 Ollama 本地模型。但 Function Calling 的 JSON 格式你必须知道，因为这是行业标准。
>
> 运行 Cell 9gpup7a0edc。（运行）这段代码完整演示了 Function Calling 的三步流程：
>
> **Step 1：工具定义。** 看输出——`get_weather` 工具，参数有 `city`（string，必填）和 `unit`（enum 约束只能选 celsius 或 fahrenheit，可选）。这个 JSON Schema 会被发给 LLM。
>
> **Step 2：LLM 返回 tool_calls。** 注意看——`content: null`！这次 LLM 不是在说话，而是在调用工具。`tool_calls` 数组里有调用 ID `call_abc123`、函数名 `get_weather`、参数 `{"city": "Tokyo", "unit": "celsius"}`。
>
> **Step 3：工具结果回传。** `role: "tool"`（注意不是 assistant 也不是 user），通过 `tool_call_id` 关联回 Step 2 的调用。`content` 是天气数据 JSON。
>
> **最终：** LLM 看到工具结果后，输出自然语言回复「东京现在天气晴朗，气温 22°C。」——这次 `content` 有值了！
>
> 大家记住这个模式：**定义 → 调用 → 回传 → 最终回复**。不管用什么框架（LangChain、LlamaIndex），底层都是这个流程。

👀 输出要点
- Cell tool-class：Tool 数据类定义（无输出，代码定义）
- Cell 9gpup7a0edc 四段输出：
  - Step 1：工具定义 JSON（`get_weather`，参数 `city` + `unit`）
  - Step 2：LLM 响应（`content: null`，`tool_calls` 含 `call_abc123`）
  - Step 3：工具结果（`role: "tool"`，温度 22°C，Clear sky）
  - Final：最终回复（`content: "东京现在天气晴朗, 气温 22°C。"`）

❓ 预判问题
- **Q：为什么 Step 2 的 content 是 null？**
  A：因为 LLM 这次选择调用工具而不是直接回答。调用工具时不需要生成文本内容，所以 content 为空。这是 Function Calling 协议的规定。
- **Q：tool_call_id 有什么用？**
  A：用于关联请求和响应。如果 LLM 一次调用多个工具（并行调用），每个调用有不同的 id，结果回传时要一一对应。
- **Q：本 notebook 为什么不直接用 Function Calling？**
  A：因为 Ollama 本地模型对 Function Calling 的支持不够稳定。文本解析方式更通用，兼容所有 LLM。理解原理后，切换到 Function Calling 只是改调用方式，逻辑完全一样。

➡️ 转场

> Function Calling 的格式理解了。下面我们看 JSON Schema 的参数定义细节，然后动手实现四个真实工具。

---

## 第三段：四个工具实现 + 测试验证（Cell yctzavkcfvd ~ 8igrjqox0qg）

📍 浏览 Cell yctzavkcfvd（JSON Schema 速查），运行 Cell tool-implementations（计算器），运行 Cell tool-search（搜索），运行 Cell tool-datetime（日期时间），运行 Cell tool-weather（天气），运行 Cell tools-collection（工具注册），运行 Cell 8igrjqox0qg（OpenAI 格式输出）。浏览 Cell 9w11oo2miwn（Function Calling vs ReAct 对比）

⏱ 时间分配：15 分钟

🎯 本段目标
- 看到四个工具的具体实现和真实 API 调用
- 理解 JSON Schema 参数定义的细节
- 验证每个工具独立可用
- 理解工具注册表和 OpenAI 格式转换

🗣 讲课话术

> 先快速看一下 Cell yctzavkcfvd 的 JSON Schema 速查表。这是给 LLM 看的参数说明标准格式。六种基本类型——string、number、integer、boolean、array、object。重点记住两个关键字段：`required`（哪些参数必填）和 `description`（给 LLM 看的参数说明）。**description 写得越清楚，LLM 填参数越准确。**
>
> 好，现在逐个运行四个工具。
>
> **工具一：计算器。** 运行 Cell tool-implementations。（运行）看代码——内部用 `eval()` 但做了安全限制：(1) 只允许安全的数学函数（sqrt、sin、cos、log 等）；(2) 只允许安全字符，禁止 `import`、`exec` 等危险操作。测试结果：`2 + 3 * 4 = 14`（运算优先级正确），`sqrt(16) = 4.0`，`sin(pi/2) = 1.0`。
>
> **工具二：网络搜索。** 运行 Cell tool-search。（运行）这个用的是 DuckDuckGo 的 Instant Answer API——免费、不需要 API Key。看测试结果——搜索 "Python programming language" 返回了一大段摘要：Python 是高级通用编程语言，由 Guido van Rossum 创建……还有 Pydoc、NumPy 等相关话题。这是真实的网络搜索！
>
> **工具三：日期时间。** 运行 Cell tool-datetime。（运行）返回当前日期、时间、星期几、第几周、时间戳。看输出——Date 是今天的日期，Day of Week 是星期几。这个工具很简单但很实用，LLM 不知道「现在几点」，需要这个工具告诉它。
>
> **工具四：天气查询。** 运行 Cell tool-weather。（运行）这个用了两步 API 调用——先用 Open-Meteo 的 geocoding API 把城市名转坐标（"Tokyo" → 纬度 35.68、经度 139.69），再用天气 API 查当前天气。看结果——Tokyo, Japan，温度、湿度、风速、天气状况全有。完全免费，不需要 API Key！
>
> 运行 Cell tools-collection 注册所有工具。（运行）四个工具都注册成功。
>
> 最后运行 Cell 8igrjqox0qg 看 OpenAI 格式。（运行）`calculator_tool.to_openai_format()` 输出的就是 Step 1 发给 API 的工具定义 JSON。底部显示 4 个工具已转换。实际调用 API 时，就是把这个列表传给 `tools` 参数。
>
> Cell 9w11oo2miwn 有一个重要对比——**Function Calling vs ReAct 文本解析**的完整代码对比。Function Calling 用 `client.chat.completions.create(tools=...)` 直接调用，模型返回结构化 JSON。ReAct 用 System Prompt 描述工具格式，模型返回自由文本，我们用正则解析。选择建议：用 OpenAI/DashScope API 选 Function Calling 更可靠；用 Ollama 本地模型选 ReAct 更通用；教学先学 ReAct 理解原理。

👀 输出要点
- Cell tool-implementations（计算器测试）：
  - `2 + 3 * 4 = 14`
  - `sqrt(16) = 4.0`
  - `sin(pi/2) = 1.0`
- Cell tool-search（搜索测试）：
  - Summary：Python is a high-level, general-purpose programming language...
  - Related Topics：Pydoc、NumPy 等
- Cell tool-datetime（日期时间测试）：
  - Date: 2026-03-15（实际日期）
  - Day of Week: Sunday（实际星期）
- Cell tool-weather（天气测试）：
  - Weather in Tokyo, Japan: Clear sky, Temperature: 5.1°C, Humidity: 72%, Wind Speed: 3.8 km/h
- Cell tools-collection：4 个工具注册成功
- Cell 8igrjqox0qg：calculator 和 weather 的 OpenAI JSON 格式 + 共 4 个工具已转换

❓ 预判问题
- **Q：calculator 用 eval() 安全吗？**
  A：教学环境做了两层防护——安全命名空间（只暴露数学函数）+ 字符白名单。但生产环境绝对不能用 eval！应该用 `ast.literal_eval` 或 `sympy` 这样的数学库。
- **Q：DuckDuckGo API 为什么有时搜不到结果？**
  A：Instant Answer API 只返回「即时答案」——维基百科摘要和相关话题。对于非常具体或实时的查询可能返回空。真正的搜索引擎用 Bing API 或 Google API 更好，但需要付费。
- **Q：Open-Meteo API 免费有限制吗？**
  A：个人和非商业用途免费，每天约 10000 次调用限额。教学完全够用。
- **Q：能不能自己加工具？**
  A：当然可以！写一个 Python 函数，创建一个 Tool 实例，加到 ALL_TOOLS 列表里就行。后面的练习环节（Cell exercise）就是加一个单位转换工具。

➡️ 转场

> 四个工具都验证通过了，工具注册表也建好了。接下来是最核心的部分——构建 ReAct Agent，让 LLM 自己决定什么时候调什么工具。先休息 5 分钟。

---

## 休息 + 回顾（第 40-45 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. ReAct = Reasoning + Acting。Agent 的工作循环是 Thought → Action → Observation → ... → Final Answer。本 notebook 与 Ch12 的区别在于——这次用**真实 LLM** 驱动推理，用**真实 API** 执行工具。
2. 工具系统有四个要素：名称、描述、JSON Schema 参数、执行函数。`description` 是给 LLM 看的 API 文档，写得越清楚 LLM 填参数越准确。
3. Function Calling 是行业标准的工具调用协议（定义 → 调用 → 回传），本 notebook 用的是更通用的 ReAct 文本解析方式（正则提取 Action/Action Input），两种方式的核心循环逻辑完全一致。

**下一段预告：** 接下来是本节课最核心的部分——ReActAgent 类的完整实现。200 行代码实现一个能自主推理和使用工具的 AI Agent。

---

## 第四段：ReAct Agent 核心实现（Cell agent-header ~ agent-class）

📍 浏览 Cell agent-header（Markdown 流程说明），运行 Cell agent-class（ReActAgent 完整实现）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解 ReActAgent 类的五大组件：System Prompt、工具描述生成、响应解析、工具执行、主循环
- 掌握 System Prompt 中工具描述的格式和作用
- 理解正则表达式解析 LLM 输出的方法
- 看到 Agent 创建成功

🗣 讲课话术

> Cell agent-header 总结了核心流程——四步：(1) 接收问题，构建包含工具描述的 prompt；(2) LLM 思考，输出 Thought + Action；(3) 执行工具，获取 Observation；(4) 循环，直到 LLM 给出 Final Answer。
>
> 运行 Cell agent-class。（运行）看到 `[OK] ReAct Agent 创建成功!` 就好。但我们要仔细过一遍这 200 行代码。
>
> **第一部分：SYSTEM_PROMPT。** 这是发给 LLM 的系统提示词，告诉它：(1) 你是一个能使用工具的 AI 助手；(2) 可用工具列表（由 `{tool_descriptions}` 占位符填入）；(3) 必须遵循严格格式——用 `Thought:` 开头思考，用 `Action:` 指定工具名，用 `Action Input:` 传 JSON 参数；(4) 得到足够信息后用 `Final Answer:` 回答。
>
> 打个比方：SYSTEM_PROMPT 就像给新员工的工作手册——「你有这些工具可以用，每次先想清楚，写明你要用哪个工具，传什么参数，拿到结果后再回答客户。」
>
> **第二部分：`_get_tool_descriptions()`。** 把所有工具的名称、描述、参数拼成文本。这段文本会填入 SYSTEM_PROMPT 的 `{tool_descriptions}` 位置。
>
> **第三部分：`_parse_response()`——这是最关键的解析逻辑。** 用正则表达式从 LLM 的自由文本中提取结构化信息：
> - `r'Thought:\s*(.+?)(?=Action:|Final Answer:|$)'` → 提取思考内容
> - `r'Final Answer:\s*(.+?)$'` → 提取最终答案
> - `r'Action:\s*([\w_]+)'` → 提取工具名称
> - `r'Action Input:\s*(.+?)(?=Thought:|...)'` → 提取参数 JSON
>
> 如果解析到 `Final Answer`，循环结束。如果解析到 `Action`，执行工具。如果都解析不到，让 LLM 重试。
>
> **第四部分：`run()` 主循环。** 这是 Agent 的心脏。伪代码是：
> ```
> for step in range(max_steps):
>     response = llm.chat(messages)        # 调用真实 LLM
>     parsed = parse_response(response)    # 解析输出
>     if parsed.final_answer:              # 有最终答案？
>         return final_answer              # 结束
>     if parsed.action in tools:           # 有合法工具调用？
>         observation = tool.run(...)      # 执行工具
>         messages.append(observation)     # 把结果加回对话
>     # 继续循环...
> ```
>
> 注意 `temperature=0.1`——Agent 场景需要低温度，因为我们要 LLM 严格遵循格式，不要太有创造力。
>
> 还有 `max_steps=10` 安全阀——防止无限循环。如果 10 步还没答案，强制终止。
>
> 最后看错误处理：工具不存在时返回错误消息；无法解析时要求 LLM 重新按格式回答。这种「容错 + 重试」机制在生产系统中非常重要。
>
> 大家有没有注意到一个设计选择？工具执行结果是以 `{"role": "user", "content": f"Observation: {observation}"}` 的形式加入对话的——也就是说，Observation 是以「用户消息」的形式告诉 LLM 的。为什么不用 `role: "tool"`？因为 ReAct 文本解析模式下，LLM 不一定支持 tool 角色，用 user 角色更通用。

👀 输出要点
- Cell agent-class：`[OK] ReAct Agent 创建成功!`
- 代码结构：
  - `SYSTEM_PROMPT`：包含工具描述模板和格式要求
  - `_parse_response()`：4 个正则表达式提取 Thought/Action/Action Input/Final Answer
  - `run()`：最大 10 步的 while 循环，调用 LLM → 解析 → 执行工具 → 回传

❓ 预判问题
- **Q：为什么 temperature 设 0.1 而不是 0？**
  A：0.1 比 0 稍微有一点随机性，能避免某些模型在 temperature=0 时的退化行为（如重复输出）。但本质上我们希望 Agent 输出尽量确定、格式规范。
- **Q：如果 LLM 输出的格式不对怎么办？**
  A：代码里有兜底——如果既没有 Final Answer 也没有 Action，会发一条消息要求 LLM 重新按格式回答。这就是「重试机制」。但如果模型能力太弱（比如很小的模型），可能反复格式错误直到 max_steps 用完。
- **Q：为什么 Observation 用 user 角色而不是 tool 角色？**
  A：ReAct 文本解析模式下不走 Function Calling 协议，LLM 不一定支持 `role: "tool"`。用 `role: "user"` 加 `Observation:` 前缀是最通用的做法。如果用 Function Calling 则应该用 `role: "tool"` + `tool_call_id`。
- **Q：history 列表有什么用？**
  A：记录 Agent 每一步的 Thought、Action、Observation，用于后面的可视化和调试。生产系统还可以用 history 做日志和回放。

➡️ 转场

> Agent 的代码看完了，但「代码好不好」要看效果。下面我们用五个不同类型的问题来测试这个 Agent——数学、时间、天气、搜索、复合推理。

---

## 第五段：Agent 测试——五个不同场景（Cell test-header ~ test-complex）

📍 浏览 Cell test-header（Markdown），依次运行 Cell test-math、test-datetime、test-weather、test-search、test-complex

⏱ 时间分配：15 分钟（每个测试约 3 分钟）

🎯 本段目标
- 观察 Agent 在不同类型问题上的推理过程
- 理解单步推理 vs 多步推理的区别
- 发现 Agent 的真实行为（包括错误恢复）
- 体会真实 LLM 驱动 Agent 的效果和局限

🗣 讲课话术

> **测试 1：数学计算。** 运行 Cell test-math。（运行）问题是「What is the square root of 144 plus 25?」看 Agent 的思考过程——Step 1，LLM 输出 Thought「我需要计算 sqrt(144) + 25」，然后 Action: calculator，Action Input: `{"expression": "sqrt(144) + 25"}`。
>
> 大家注意一个有趣的现象——LLM 在同一个回复里既输出了 Action 又输出了 Final Answer: 37。这说明它「想快了」，一步到位给了答案。Agent 的解析逻辑优先检测 Final Answer，所以直接返回了 37。这其实是模型足够强的表现——它在心算就算出来了。
>
> **测试 2：日期时间。** 运行 Cell test-datetime。（运行）问题是「今天星期几？」Step 1：LLM 调用 get_datetime 工具，不传参数。Observation 返回完整的日期时间信息——Date、Time、Day of Week 都有。Step 2：LLM 看到 Observation 里 Day of Week 是 Sunday，给出 Final Answer「Today is Sunday.」——两步完成，很标准的 ReAct 循环。
>
> 注意这个问题如果不调用工具，LLM 是回答不了的——它不知道「现在」是什么时候。这就是为什么我们需要工具。
>
> **测试 3：天气查询。** 运行 Cell test-weather。（运行）问题是东京天气。Step 1：LLM 调用 get_weather，传 `{"city": "Tokyo"}`。但看——这次 LLM 在同一个回复里又输出了 Action 和 Final Answer！它说 Final Answer 是「The current weather in Tokyo is [weather information]」——这是一个**占位符式回答**，说明 LLM「着急了」，没等到真正的 Observation 就抢先回答了。
>
> 这是真实 LLM Agent 的常见问题！模型有时会在一个回复里同时输出 Action 和 Final Answer，导致 Final Answer 不准确。解决方案：(1) 在 prompt 里加更强的格式约束；(2) 解析时如果同时检测到 Action 和 Final Answer，优先执行 Action。当前代码优先取 Final Answer，所以返回了不完整的回答。
>
> **测试 4：网络搜索。** 运行 Cell test-search。（运行）问题是「什么是 Python？」这次 LLM 判断——它自己就知道什么是 Python，不需要搜索！所以直接给出了 Final Answer，一步完成，没有调用任何工具。这说明 LLM 有**自主判断能力**——该用工具就用，不该用就直接回答。
>
> **测试 5：复合问题（重点）。** 运行 Cell test-complex。（运行）问题包含两部分——「如果今天是 Monday，等 15 天后是星期几？另外计算 15 * 7。」
>
> 看 Agent 的推理过程！Step 1：LLM 试图用 calculator 算 `15 % 7`（取余）。但 Observation 返回 Error——因为 calculator 的字符白名单不包含 `%`！Step 2：LLM 再试一次 `15 % 7`，还是失败。Step 3：LLM 学聪明了！它改用心算——「7 * 2 = 14，15 - 14 = 1，所以余 1 天」——然后用 calculator 算 `15 * 7`，得到 105。Step 4：整合答案——「15 天后是 Tuesday，15 * 7 = 105」。
>
> 这个测试展示了 Agent 最精彩的部分——**错误恢复能力**！工具执行失败，LLM 不是卡住了，而是调整策略，用另一种方式解决问题。这就是真正的推理能力。
>
> 同时也暴露了一个 bug——calculator 的字符白名单忘了加 `%`。这就是测试的价值。

👀 输出要点
- Cell test-math：1 步完成，sqrt(144) + 25 = 37（LLM 心算直接给答案）
- Cell test-datetime：2 步完成
  - Step 1：Action: get_datetime → Observation: Date 2026-03-15, Day of Week: Sunday
  - Step 2：Final Answer: Today is Sunday.
- Cell test-weather：1 步完成但答案不完整（LLM 提前给了 Final Answer 占位符）
  - 暴露问题：LLM 在同一回复里同时输出 Action 和 Final Answer
- Cell test-search：1 步完成，LLM 判断不需要工具，直接回答 Python 介绍
- Cell test-complex：4 步完成（最精彩）
  - Step 1-2：calculator 不支持 `%`，两次报错 `Error: Invalid characters`
  - Step 3：LLM 自行推理余数，用 calculator 算 `15 * 7 = 105`
  - Step 4：Final Answer: Tuesday, 15 * 7 = 105

❓ 预判问题
- **Q：为什么天气测试的 Final Answer 不完整？**
  A：LLM 在同一个回复里既调用了工具又给了 Final Answer（带占位符）。`_parse_response()` 优先返回 Final Answer，所以没等到真正的 Observation。可以通过改进解析逻辑来修复——如果同时检测到 Action 和 Final Answer，优先执行 Action。
- **Q：为什么搜索测试没有调用 web_search？**
  A：LLM 自己就知道什么是 Python（训练数据里有大量相关内容）。只有 LLM 不确定或需要实时信息时才会调用搜索工具。这是 Agent 的自主判断能力。
- **Q：复合问题为什么试了两次 `%` 才放弃？**
  A：LLM 的「记忆」是对话上下文。第一次报错后，它收到 Observation: Error，但还是想试试同样的方法。第二次还是失败后，它才意识到需要换策略。这其实很像人类——同一个方法试两次失败才会换思路。
- **Q：每个测试花了多少步？**
  A：数学 1 步，时间 2 步，天气 1 步，搜索 1 步，复合 4 步。简单问题 1-2 步，复合问题需要更多步。max_steps=10 对这些场景绰绰有余。

➡️ 转场

> 五个测试看完了。Agent 表现很好——能调工具、能自主判断、遇到错误还能恢复。下面看可视化流程图和总结练习。

---

## 第六段：可视化 + 总结 + 练习（Cell viz-header ~ exercise）

📍 运行 Cell viz-flow（ReAct 流程可视化），浏览 Cell summary-header（总结），运行 Cell exercise（练习：单位转换工具）

⏱ 时间分配：5 分钟

🎯 本段目标
- 通过可视化巩固 ReAct 流程的全局观
- 回顾本节四个核心收获
- 了解扩展练习方向

🗣 讲课话术

> 运行 Cell viz-flow。（运行）这张图把 ReAct Agent 的完整流程画出来了。从上到下看：
>
> 1. 用户问题（绿色框）：「东京现在天气怎么样？」
> 2. LLM 推理（紫色框）：真实的 Ollama/DashScope/OpenAI 在思考
> 3. 思考（黄色框）：「需要调用天气工具」
> 4. 行动（红色框）：`get_weather`，参数 `{"city": "Tokyo"}`
> 5. 工具执行（橙色框）：Open-Meteo API 调用
> 6. 观察（蓝色框）：温度 22°C，晴天
> 7. 循环箭头：回到 LLM 推理，直到得出最终答案
> 8. 最终答案（绿色框）：「东京现在 22°C，晴天」
>
> 输出下方还有四个关键点——(1) 真实 LLM 推理；(2) 真实 API 调用；(3) 迭代式思考；(4) 可扩展工具系统。
>
> Cell summary-header 总结了三大收获：
> 1. **ReAct 模式**：Thought → Action → Observation 循环，真实 LLM 驱动
> 2. **工具系统**：统一接口、真实 API、JSON Schema 参数规范
> 3. **LLM 后端**：Ollama/DashScope/OpenAI 统一接口，灵活切换
>
> 还有三个扩展练习建议：(1) 添加翻译工具；(2) 用 JSON mode 改进解析可靠性；(3) 添加记忆（让 Agent 记住之前的对话）。
>
> 最后运行 Cell exercise 看练习代码。（运行）这是一个单位转换工具——支持温度（C/F/K）、长度（m/ft/km/mi）、重量（kg/lb）。测试结果：`100 c = 212.00 f`，`1 km = 0.62 mi`。
>
> 大家如果想自己加到 Agent 里，只需要三步：(1) 写函数（已经有了）；(2) 创建 Tool 实例（定义名称、描述、参数 Schema）；(3) 加到 ALL_TOOLS 列表里，重新创建 Agent。

👀 输出要点
- Cell viz-flow：ReAct 流程图（7 个彩色框 + 循环箭头）+ 4 个关键点总结
- Cell exercise：
  - `100 c = 212.00 f`（摄氏转华氏）
  - `1 km = 0.62 mi`（公里转英里）

❓ 预判问题
- **Q：如何把练习的 unit_converter 加到 Agent 里？**
  A：创建 `converter_tool = Tool(name="unit_converter", description="...", parameters={...}, func=unit_converter)`，然后 `ALL_TOOLS.append(converter_tool)`，最后 `agent = ReActAgent(llm, ALL_TOOLS)` 重新创建 Agent。
- **Q：JSON mode 是什么？**
  A：部分 LLM API 支持 `response_format={"type": "json_object"}`，强制模型只输出合法 JSON。这比正则解析更可靠，但不是所有模型都支持。
- **Q：添加记忆怎么做？**
  A：最简单的方式是保留对话历史（messages 列表），下次提问时把之前的 messages 也传进去。更高级的做法是用向量数据库存储历史对话，按相似度检索相关上下文。

➡️ 转场

> 本节课的核心内容到这里就讲完了。最后做一个简短的总结回顾。

---

## 休息 + 总结回顾（第 80-85 分钟）

⏱ 时间分配：5 分钟

**三句话总结全课：**

1. **ReAct Agent 的核心是一个循环**：LLM 思考（Thought）→ 选择工具（Action）→ 执行并观察（Observation）→ 再思考 → ... → 最终答案。本节课用真实 LLM 驱动了这个循环，四个工具都调用了真实 API（DuckDuckGo、Open-Meteo、系统时间、安全 eval）。
2. **工具系统 = 函数 + 说明书**：每个工具是一个 Python 函数 + JSON Schema 参数描述。`description` 写得好不好直接决定 LLM 能否正确调用。Function Calling（结构化 JSON）和 ReAct 文本解析（正则提取）是两种主流实现方式。
3. **真实 Agent 不完美但有韧性**：我们看到了 LLM「提前回答」（天气测试）、「重复犯错后调整策略」（复合问题 `%` 错误恢复）等真实行为。理解这些局限才能构建可靠的 Agent 系统——需要格式约束、错误处理、max_steps 安全阀。

**下节课预告：** App2 RAG System——给 LLM 装上「外部大脑」，实现检索增强生成。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，ReAct 概念导入，运行环境准备 + LLM 后端配置 | intro-cell ~ setup-llm |
| 8 | Tool 类定义 + Function Calling 三步流程 + JSON 格式演示 | tools-header ~ 9gpup7a0edc |
| 25 | JSON Schema 速查 + 四个工具实现 + 测试 + 工具注册 + ReAct vs FC 对比 | yctzavkcfvd ~ 9w11oo2miwn |
| 40 | **休息** | -- |
| 45 | ReActAgent 类：System Prompt + 正则解析 + 主循环 | agent-header ~ agent-class |
| 60 | 五个测试场景：数学 / 时间 / 天气 / 搜索 / 复合推理 | test-header ~ test-complex |
| 75 | 可视化流程图 + 总结 + 单位转换练习 | viz-header ~ exercise |
| 80 | **休息 + 总结回顾** | -- |

---

## 附录 B：关键数据快速参考

### ReAct 循环

```
问题 → Thought → Action → Observation → Thought → ... → Final Answer
```

### 工具系统四要素

| 要素 | 说明 | 示例 |
|:---|:---|:---|
| name | 唯一标识符 | `calculator` |
| description | LLM 用来决定何时使用 | `"Perform mathematical calculations..."` |
| parameters | JSON Schema 格式 | `{"type": "object", "properties": {...}, "required": [...]}` |
| func | 实际执行函数 | `calculator(expression: str) -> str` |

### 四个工具测试输出

| 工具 | 测试输入 | 返回值 |
|:---|:---|:---|
| calculator | `2 + 3 * 4` | `14` |
| calculator | `sqrt(16)` | `4.0` |
| calculator | `sin(pi/2)` | `1.0` |
| web_search | `"Python programming language"` | DuckDuckGo 摘要 + 相关话题 |
| get_datetime | （无参数） | 当前日期、时间、星期、周数、时间戳 |
| get_weather | `"Tokyo"` | Tokyo, Japan: Clear sky, 5.1°C, 72% humidity |

### Function Calling 三步流程

| 步骤 | 角色 | 关键字段 |
|:---|:---|:---|
| Step 1: 工具定义 | 开发者 → API | `tools: [{type: "function", function: {name, description, parameters}}]` |
| Step 2: LLM 调用 | API 响应 | `content: null, tool_calls: [{id, function: {name, arguments}}]` |
| Step 3: 结果回传 | 系统 → LLM | `role: "tool", tool_call_id, content: "结果JSON"` |

### Agent 测试结果汇总

| 测试 | 问题类型 | 步数 | 工具调用 | 关键观察 |
|:---|:---|:---|:---|:---|
| test-math | 数学计算 | 1 | calculator（心算跳过） | LLM 同时输出 Action + Final Answer |
| test-datetime | 实时信息 | 2 | get_datetime | 标准 ReAct 两步循环 |
| test-weather | API 查询 | 1 | get_weather（未实际执行） | LLM 提前给占位符答案 |
| test-search | 知识查询 | 1 | 无（自主判断不需要） | LLM 自主决定不用工具 |
| test-complex | 复合推理 | 4 | calculator（2 次失败 + 1 次成功） | 错误恢复：`%` 不支持 → 心算替代 |

### ReActAgent 核心参数

| 参数 | 值 | 说明 |
|:---|:---|:---|
| temperature | 0.1 | 低温度保证格式稳定 |
| max_steps | 10 | 防无限循环安全阀 |
| 解析方式 | 正则表达式 | 提取 Thought/Action/Action Input/Final Answer |
| Observation 传递 | `role: "user"` | 通用兼容，非 Function Calling 的 `role: "tool"` |

### LLM 后端对比

| 后端 | 优点 | 缺点 | 适用场景 |
|:---|:---|:---|:---|
| Ollama（本地） | 免费、隐私、离线可用 | 需要 GPU/CPU 资源 | 个人练习、敏感数据 |
| DashScope（阿里云） | 稳定、快、模型多 | 需要 API Key、按量付费 | 教学演示、生产环境 |
| OpenAI | 模型能力最强 | 贵、需要海外网络 | 高质量需求 |

---

## 附录 C：应急预案

### 场景 1：LLM 后端连接失败

**症状：** `[X] 请配置 LLM 后端（任选其一）`

**应对：**
1. **Ollama**：确认 Ollama 服务已启动（`ollama serve`），模型已下载（`ollama pull qwen3:4b`）
2. **DashScope**：确认环境变量 `DASHSCOPE_API_KEY` 已设置，API Key 有效
3. **OpenAI**：确认环境变量 `OPENAI_API_KEY` 已设置，网络可访问
4. 如果全部失败：可以用 Cell setup-llm 里的手动配置方式，取消注释其中一种
5. 最差情况：跳过实际运行，用预跑的输出截图讲解 Agent 行为

### 场景 2：网络 API 调用超时

**症状：** 天气工具或搜索工具返回 `Search error: ReadTimeout` 或 `Weather API error: ConnectionError`

**应对：**
1. 检查网络连接，确认能访问 `api.duckduckgo.com` 和 `api.open-meteo.com`
2. 如果在公司/学校网络，可能有代理限制——尝试设置 `HTTP_PROXY` 环境变量
3. 工具调用失败不影响 Agent 核心逻辑的教学——告诉学生「Observation 返回 Error，Agent 会尝试其他策略」
4. 计算器和日期时间工具不依赖网络，可以先用这两个工具演示

### 场景 3：LLM 输出格式不规范导致解析失败

**症状：** Agent 反复输出 `Please respond in the correct format...` 直到 max_steps

**应对：**
1. 这是小模型的常见问题——模型能力不足以严格遵循 ReAct 格式
2. 解决方案 A：换更强的模型（如 qwen3:8b 或 qwen-plus）
3. 解决方案 B：简化 SYSTEM_PROMPT，减少格式要求
4. 教学时正好作为反面案例——解释为什么 Agent 需要足够强的 LLM 基座

### 场景 4：天气测试返回占位符答案

**症状：** `The current weather in Tokyo is [weather information]`

**应对：**
1. 这是已知行为——LLM 在同一回复里同时输出 Action 和 Final Answer
2. 向学生解释这是真实 Agent 的常见问题，不是 bug
3. 讨论解决方案：改进 `_parse_response()` 逻辑——如果同时检测到 Action 和 Final Answer，优先执行 Action
4. 这其实是一个很好的教学案例——真实系统的调试过程

### 场景 5：复合测试（test-complex）步数过多或无限循环

**症状：** Agent 在 `15 % 7` 上反复失败超过 3 次

**应对：**
1. 不同 LLM 表现不同——有的模型 2 次失败就会换策略，有的需要更多次
2. 如果超过 5 步还在循环，可以手动中断（Kernel → Interrupt）
3. 解释 `max_steps=10` 安全阀的作用——最终会强制终止
4. 可以在课前给 calculator 的字符白名单加上 `%`，避免这个问题

### 场景 6：时间不够

**可以跳过的内容（按优先级）：**
1. Cell yctzavkcfvd（JSON Schema 速查表）——口头带过即可
2. Cell 9w11oo2miwn（Function Calling vs ReAct 对比）——课后阅读
3. Cell 8igrjqox0qg（OpenAI 格式转换）——简化为一句话
4. Cell viz-flow（可视化流程图）——用白板手画替代

**不能跳过的内容：**
1. Cell setup-imports + setup-llm——环境必须能跑
2. Cell tool-class——理解工具抽象
3. Cell 9gpup7a0edc——Function Calling 三步 JSON 格式
4. 至少两个工具实现（calculator + weather）——看到真实 API 调用
5. Cell agent-class——Agent 核心代码
6. 至少两个测试（test-datetime + test-complex）——看到标准循环 + 错误恢复